Model

In [1]:
import anndata
import numpy as np
import pandas as pd
from scipy.sparse import spmatrix
import scanpy as sc
from sklearn.preprocessing import StandardScaler
from celltypist import models

import logging
logger = logging.getLogger("Data-Preprocessing")

Data Preprocessing

In [2]:
marker_data = pd.read_csv("/home/lagandua/projects/sven/pineapple/Cell_marker_Human.csv", sep=",", index_col=0, encoding='unicode_escape')

data_path = "./data/global.h5ad"
raw = anndata.read_h5ad(data_path)
logger.info("🍳 Preparing data before training")

old_to_new_class_map = {}
with open("/home/lagandua/projects/sven/pineapple/cell_mapping2.txt", "r") as file:
    for line in file:
        line = line.strip()
        split_line = line.split(":")
        old_classes = split_line[1].split(",")
        new_class = split_line[0]
        for c in old_classes:
            old_to_new_class_map[c] = new_class

🍳 Preparing data before training


In [3]:

# Create new class labels
# label = "sven_classes"
# raw.obs[label] = [
#     old_to_new_class_map[c] for c in raw.obs["Predicted_labels_CellTypist"]
# ]

label = "Predicted_labels_CellTypist"

In [4]:
# remove classes for which I do not have any marker genes
# load dictionary
import pickle
with open('marker_data_cell_name_mapping.pkl', 'rb') as f:
    source_to_target = pickle.load(f)

counter = 0
valid_cell_names = []
valid_classes = []
for key in source_to_target.keys():
    if source_to_target[key] == []:
        counter += 1
    else:
        valid_cell_names.extend(source_to_target[key])
        valid_classes.append(key)
print(counter, "classes have no marker genes")

raw = raw[raw.obs[label].isin(valid_classes)]

36 classes have no marker genes


In [5]:
# Value counts of new class labels
logger.info(f"🍳 Total classes {len(raw.obs[label].unique())}")
logger.info("🍳 Value counts of new class labels"
            f"\n{raw.obs[label].value_counts()}")

🍳 Total classes 55
🍳 Value counts of new class labels
Predicted_labels_CellTypist
Cytotoxic T cells                               36883
Memory B cells                                  31194
Helper T cells                                  25761
Macrophages                                     24071
Tem/Effector helper T cells                     21015
Classical monocytes                             18388
Naive B cells                                   13665
Tem/Effector cytotoxic T cells                   9749
Regulatory T cells                               9626
Tcm/Naive cytotoxic T cells                      9125
Plasma cells                                     8882
T cells                                          6755
gamma-delta T cells                              6745
Follicular helper T cells                        6470
Type 1 helper T cells                            5648
Non-classical monocytes                          4178
MAIT cells                                       3854


In [6]:

# Only keep the classes that have 2000 or more cells
# min_cells_per_class = 2000
classes = raw.obs[label].value_counts().index
# classes = classes[classes >= min_cells_per_class].index
# logger.info(f"🍳 {len(set(classes))} Classes with {min_cells_per_class} or more cells"
#             f"\n{classes}")

In [7]:
# Create a gene filter with scanpy's highly_variable_genes
n_top_genes = 10000
gene_filer = sc.pp.highly_variable_genes(raw, flavor="seurat_v3", n_top_genes=n_top_genes, inplace=False)

In [8]:
indata = raw.X[raw.obs[label].isin(classes)]
genes = raw.var_names
labels = raw.obs[raw.obs[label].isin(classes)][label]

if isinstance(indata, pd.DataFrame):
    indata = indata.values
elif isinstance(indata, spmatrix):
    indata = indata.toarray()

labels = np.array(labels)
genes = np.array(genes)

In [9]:
# Filter genes based on the celltypist model
# models.models_path = "/home/lagandua/projects/sven/pineapple/celltypist/.cache"
# ct_model = models.Model.load(model = 'Immune_All_Low.pkl')
# chosen_genes = np.array(
#         list(set(ct_model.classifier.features).intersection(set(raw.var_names)))
#     )
# keep_model = np.isin(raw.var_names, chosen_genes)

# Filter genes based on the highly variable genes
keep_model = gene_filer["highly_variable"].values
flag = indata.sum(axis = 0) == 0
if flag.sum() > 0:
    logger.info(f"✂️ {flag.sum()} non-expressed are filtered out")
    keep = (~flag) & keep_model
    genes = genes[keep]
logger.info(f"🔬 Input data has {indata.shape[0]} cells and {(keep).sum()} genes")

✂️ 4226 non-expressed are filtered out
🔬 Input data has 264573 cells and 10000 genes


In [10]:
# First get a common marker gene expression 
all_marker_genes = set(marker_data.marker[marker_data.cell_name.isin(valid_cell_names)])

# Get the common genes between the marker genes and the genes in the data (intersection)
marker_genes = set(genes).intersection(all_marker_genes)

# Get the indices of the common genes
marker_genes_indices = np.array([True if gene in marker_genes else False for i, gene in enumerate(genes)])

# Make sure that the indices are correct
assert marker_genes_indices.sum() == len(marker_genes)

In [11]:
# Scaler (this takes forever to run)
# Set to False if you are want to use the data for VAE with ZINB loss
# Set to True if you are want to use the data for VAE with MSE loss
scale_data = False
if scale_data:
    logger.info(f"⚖️ Scaling input data")
    scaler = StandardScaler(with_mean = True)
    indata = scaler.fit_transform(indata[:, keep] if keep.sum() > 0 else indata)
    indata[indata > 10] = 10
else:
    logger.info(f"⚖️ Not scaling the input data")
    indata = indata[:, keep] if keep.sum() > 0 else indata

⚖️ Not scaling the input data


In [12]:
# Change the data name based on the what dataset you are saving
data_name = f"low_top_{n_top_genes}_genes_sparse_markers"
# data_name = f"high_all_genes_sparse"

# Save preprocessed data into train and test sets (this also takes forever)
logger.info(f"📝 Saving preprocessed data into full, train and validation sets")

val_size = 0.1 # 10%
true_count = int(indata.shape[0] * val_size)
false_count = indata.shape[0] - true_count

val_flag = np.concatenate((np.full(true_count, True), np.full(false_count, False)))
np.random.shuffle(val_flag)

np.savez(f"./data/{label}/data_{data_name}.npz", data = indata, labels = labels, genes = genes, marker_genes_indices = marker_genes_indices)
np.savez(f'./data/{label}/train_{data_name}_{int(100 * (1-val_size))}.npz', data=indata[~val_flag], labels=labels[~val_flag], genes=genes, marker_genes_indices = marker_genes_indices)
np.savez(f'./data/{label}/val_{data_name}_{int(100 * (val_size))}.npz', data=indata[val_flag], labels=labels[val_flag], genes=genes, marker_genes_indices = marker_genes_indices)
logger.info(f"DONE! 🎉")

📝 Saving preprocessed data into full, train and validation sets
DONE! 🎉
